# Python to JavaScript Code Converter 🚀

Week 4 Exercise — ported the idea from the Python-to-C++ code generator lab

Instead of converting Python to C++, this tool converts **Python to JavaScript** — which is super practical if you're a web dev who wants to port backend logic to the frontend or Node.js.

Uses OpenAI GPT + Gradio for a nice interactive UI where you can:
- Paste your Python code
- Pick a model
- Get clean JavaScript code back
- Run the Python to see the output

By **Victor Conqueror** 🇳🇬

In [ ]:
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from IPython.display import Markdown, display

In [ ]:
# Load environment variables

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set — go set am for your .env file!")

In [ ]:
# Set up the clients

openai = OpenAI()

# You can add more providers here like we did in class
# anthropic, gemini, grok, etc.

OPENAI_MODEL = "gpt-4o-mini"

models = ["gpt-4o-mini", "gpt-4o"]
clients = {"gpt-4o-mini": openai, "gpt-4o": openai}

## The System Prompt

This is the key part — we tell the LLM exactly what we want:
- Convert Python to modern JavaScript (ES6+)
- Keep the output identical
- Use clean, idiomatic JS

In [ ]:
system_prompt = """
You are an expert code converter. Your task is to convert Python code into clean, modern JavaScript (ES6+).

Rules:
1. Respond ONLY with JavaScript code. No explanations, no markdown formatting.
2. The JavaScript code must produce IDENTICAL output to the Python code when run with Node.js.
3. Use modern JS features: const/let, arrow functions, template literals, destructuring, etc.
4. Use console.log() for print statements.
5. Handle Python-specific constructs properly:
   - list comprehensions → map/filter or for loops
   - dictionaries → objects or Maps
   - f-strings → template literals
   - range() → appropriate JS equivalent
   - enumerate() → forEach with index or entries()
6. Add brief comments only where the conversion logic is non-obvious.
7. The code should be ready to run directly in Node.js.
"""

In [ ]:
def user_prompt_for(python_code):
    return f"""
Convert this Python code to modern JavaScript (ES6+).
The JavaScript must produce identical output when run in Node.js.
Respond only with JavaScript code, nothing else.

Python code:

```python
{python_code}
```
"""

In [ ]:
def messages_for(python_code):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python_code)}
    ]

In [ ]:
def convert_to_js(model, python_code):
    """Send Python code to the LLM and get JavaScript back"""
    client = clients[model]
    response = client.chat.completions.create(
        model=model,
        messages=messages_for(python_code)
    )
    reply = response.choices[0].message.content
    # Clean up any markdown formatting the model might add
    reply = reply.replace('```javascript', '').replace('```js', '').replace('```', '')
    return reply.strip()

In [ ]:
def run_python(code):
    """Execute Python code and capture the output"""
    globals_dict = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

## Sample Python Code

A practical example — a data processing function that filters, transforms, and summarizes data.
This is the kind of thing you'd want to port from a Python backend to a JS frontend.

In [ ]:
sample_python = """# Data processing pipeline — the kind of thing you'd port to JS for the frontend

students = [
    {"name": "Chidi", "score": 85, "subject": "Math"},
    {"name": "Amara", "score": 92, "subject": "Science"},
    {"name": "Emeka", "score": 67, "subject": "Math"},
    {"name": "Ngozi", "score": 78, "subject": "Science"},
    {"name": "Tunde", "score": 95, "subject": "Math"},
    {"name": "Blessing", "score": 88, "subject": "Science"},
    {"name": "Kelechi", "score": 45, "subject": "Math"},
    {"name": "Funke", "score": 73, "subject": "Science"},
]

# Filter students who passed (score >= 60)
passed = [s for s in students if s["score"] >= 60]

# Group by subject
subjects = {}
for student in passed:
    subj = student["subject"]
    if subj not in subjects:
        subjects[subj] = []
    subjects[subj].append(student)

# Calculate stats per subject
for subject, group in subjects.items():
    scores = [s["score"] for s in group]
    avg = sum(scores) / len(scores)
    top = max(group, key=lambda s: s["score"])
    print(f"{subject}: {len(group)} passed, avg={avg:.1f}, top={top['name']} ({top['score']})")

# Overall stats
all_scores = [s["score"] for s in students]
print(f"\nOverall: {len(passed)}/{len(students)} passed")
print(f"Average score: {sum(all_scores)/len(all_scores):.1f}")
print(f"Highest: {max(all_scores)}, Lowest: {min(all_scores)}")
"""

## Let's test it out!

In [ ]:
# First, let's run the Python to see what the output should look like

print("=== Python Output ===")
print(run_python(sample_python))

In [ ]:
# Now let's convert it to JavaScript

js_code = convert_to_js(OPENAI_MODEL, sample_python)
print("=== Generated JavaScript ===")
print(js_code)

## The Gradio UI

Now let's build the interactive converter — just like we did in class with the C++ converter.
Two code panels side by side, a model selector, and buttons to run/convert.

In [ ]:
css = """
.convert-btn { background: linear-gradient(135deg, #012D01, #025a02) !important; color: white !important; font-weight: bold !important; }
.run-btn { font-weight: bold !important; }
.run-btn.py { background: #306998 !important; color: white !important; }
.run-btn.js { background: #f7df1e !important; color: #333 !important; }
"""

with gr.Blocks(css=css, theme=gr.themes.Soft(), title="Python → JavaScript Converter") as ui:
    gr.Markdown("# 🐍➡️📜 Python to JavaScript Converter")
    gr.Markdown("*Paste your Python code, pick a model, and get clean modern JavaScript!*")
    
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_code = gr.Code(
                label="Python (original)",
                value=sample_python,
                language="python",
                lines=20
            )
        with gr.Column(scale=6):
            js_output = gr.Code(
                label="JavaScript (generated)",
                value="",
                language="javascript",
                lines=20
            )

    with gr.Row():
        python_run = gr.Button("▶ Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], label="Model", show_label=False)
        convert = gr.Button("🔄 Convert to JavaScript", elem_classes=["convert-btn"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_result = gr.TextArea(label="Python output", lines=6)
        with gr.Column(scale=6):
            js_note = gr.TextArea(
                label="Note", 
                lines=6, 
                value="Copy the JavaScript code and run it in Node.js or your browser console to verify the output matches!"
            )

    convert.click(fn=convert_to_js, inputs=[model, python_code], outputs=[js_output])
    python_run.click(fn=run_python, inputs=[python_code], outputs=[python_result])

ui.launch(inbrowser=True)

## Try some more examples!

Here are some ideas to test:
- Fibonacci generator
- API response parser
- String manipulation functions
- Class definitions with methods

The model handles most Python constructs well — list comprehensions, dict operations, f-strings, etc.